**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# TinyML: Neural Networks on Microcontrollers

> ⚠️ **Draft — requires a microcontroller board (e.g. Raspberry Pi Pico, ~$6) not available at authoring time.** An instructor should run each block before teaching; remove this banner after.

The [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) pipeline, driven to its destination: a keyword-spotting-style classifier running on a $6 board with 264 KB of RAM — int8 weights exported to a C array, inference in plain C, and the [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) budget respected.

## 1. Pre-requisites

[Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb), [Intro to C](../Intro_Programming/Intro_C.ipynb), [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb).

---
### 🕐 Session 1 of 3 — *The Budget & the Model* (~35 min)
**Goal:** fit the arithmetic: 264 KB RAM, no FPU worth using — design the model backwards from the board.
**Feeds into:** Session 2 (export to C).

---

💡 **Intuition.** On a microcontroller you design *backwards from the datasheet*: RAM bounds activations, flash bounds weights, clock × cycles-per-MAC bounds latency. A spectrogram CNN at int8 with ~20k parameters fits a Pico with room to spare — the [compression workshop's](../Intro_Mach_Learn/Model_Compression.ipynb) student model is already nearly deployable. Train in float, quantize post-hoc, *verify the int8 model's accuracy on the desktop first* — debugging on-device is misery you schedule around.

```python
# desktop side (runnable there — reuse Model_Compression's SpecCNN pipeline):
# 1. train tiny CNN on 32x32 spectrograms   2. per-tensor int8 quantization
# 3. export:
def export_c_array(name, tensor_q, scale):
    vals = ', '.join(str(int(v)) for v in tensor_q.flatten())
    return (f'const int8_t {name}[{tensor_q.numel()}] = {{{vals}}};\n'
            f'const float {name}_scale = {scale}f;\n')
```

---
### 🕐 Session 2 of 3 — *Inference in Plain C* (~40 min)
**Goal:** conv/relu/pool/dense as loops over int8 arrays; accumulate in int32 (the headroom rule).
**Builds on:** Session 1; [Intro to C](../Intro_Programming/Intro_C.ipynb). &nbsp; **Feeds into:** Session 3 (on-device).

---

```c
// the whole runtime is ~100 lines of this shape (no libraries, no malloc):
void conv2d_int8(const int8_t* x, const int8_t* w, const int32_t* bias,
                 int8_t* out, int H, int W, int Cin, int Cout,
                 float x_scale, float w_scale, float out_scale) {
    for (int co = 0; co < Cout; co++)
      for (int i = 0; i < H; i++)
        for (int j = 0; j < W; j++) {
          int32_t acc = bias[co];               // int32 accumulator: the headroom rule
          for (int ci = 0; ci < Cin; ci++)
            for (int di = -1; di <= 1; di++)
              for (int dj = -1; dj <= 1; dj++) {
                int ii = i+di, jj = j+dj;
                if (ii < 0 || ii >= H || jj < 0 || jj >= W) continue;
                acc += (int32_t)x[(ci*H+ii)*W+jj] * w[((co*Cin+ci)*3+di+1)*3+dj+1];
              }
          float v = acc * x_scale * w_scale / out_scale;  // requantize
          out[(co*H+i)*W+j] = (int8_t)fmaxf(-128, fminf(127, roundf(fmaxf(0, v))));
        }
}
```

**Desktop oracle before flashing:** compile this same C on your laptop, run the exported weights against 100 test spectrograms, and require bit-identical agreement with the Python int8 simulation. Only then touch the board.

---
### 🕐 Session 3 of 3 — *On the Board* (~40 min)
**Goal:** flash it, feed it the mic, measure real latency and power.
**Builds on:** Session 2.

---

```bash
# Raspberry Pi Pico (RP2040) flow:
#   pico-sdk + cmake; main.c = ADC → [Q15 spectrogram: Real_Time_DSP §1] → conv net → LED
cmake -B build && make -C build && cp build/kws.uf2 /media/RPI-RP2/
```

💡 **Intuition.** The measurements that matter on-device: **latency** per inference (toggle a GPIO around the call, read it on a scope — or `time_us_32()`), **RAM high-water** (fill the stack with a pattern, see how much got overwritten), and **energy** per inference (a USB power meter is enough). Expect tens of ms per inference at ~30 mW: a coin cell runs your classifier for weeks. The whole [compression ladder](../Intro_Mach_Learn/Model_Compression.ipynb) exists for this moment.

---
## Where next

- [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) — the front-end feeding the net.
- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — when the microcontroller runs out of steam.